# Gold Layer - Top Genres Analytics

**Purpose:** Build the `gold_top_genres` table used by Power BI to show the most common Netflix content genres.

This notebook creates the gold_top_genres table by reading Netflix category data from the Silver Delta layer, aggregating genre counts, and saving the result to Unity Catalog.

## 1. Build Gold Table: gold_top_genres

Join Silver category data, trim genre names, and count distinct titles per genre.

In [0]:
try:
    from pyspark.sql.functions import col, countDistinct, trim
    
    CATALOG = "netflix_adb_luc"
    GOLD_SCHEMA = "gold"
    
    silver_path = "abfss://silver@nextflixprojectdltluc.dfs.core.windows.net/"
    
    category_df = spark.read.format("delta").load(silver_path + "netflix_category")
    
    gold_top_genres = (
        category_df
        .where(col("listed_in").isNotNull())
        .withColumn("genre", trim(col("listed_in")))
        .where(col("genre") != "")
        .groupBy("genre")
        .agg(countDistinct("show_id").alias("total_titles"))
        .orderBy(col("total_titles").desc())
    )
    
    gold_top_genres.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.gold_top_genres")
    
    display(gold_top_genres.limit(20))
except Exception as e:
    print(f'Error: {str(e)}')
    raise e


genre,total_titles
International Movies,1927
Dramas,1623
Comedies,1113
International TV Shows,1001
Documentaries,668
TV Dramas,599
Action & Adventure,597
Independent Movies,552
TV Comedies,436
Thrillers,392


## 2. Validate Results

Query the Gold table to verify the top 20 genres.

In [0]:
%sql
SELECT *
FROM netflix_adb_luc.gold.gold_top_genres
ORDER BY total_titles DESC
LIMIT 20;

genre,total_titles
International Movies,1927
Dramas,1623
Comedies,1113
International TV Shows,1001
Documentaries,668
TV Dramas,599
Action & Adventure,597
Independent Movies,552
TV Comedies,436
Thrillers,392
